# Pyrates FEP
# Notebook 06: Adjudication and gold standard

Builds the gold standard from the full campaign. Each feedback was coded by two experts; the
questions on which they disagree go to an **informed adjudication** campaign on the FEP platform:
two arbiters (`Penny` and `Shaquille`, the pilot experts) see the two original codings and, for
each question in disagreement, pick one of the two answers, without any other possible answer. For
each question of a feedback, the gold label is:

- the experts' answer when the two experts agree;
- the arbiters' answer when both arbiters pick the same one.

A feedback enters the gold standard only when every question has a gold label: if the arbiters pick
different answers on any question, the whole feedback is left out.

FEP assigns whole games: every game holding at least one disagreement is assigned to both arbiters.
The platform receives the list of these games and the two experts' codings of their feedback; the
arbiters' codings come back in the standard FEP export format.

### Inputs:
- `campaign/2_full/full_form.json`
- `campaign/2_full/full_campaign_assignment.csv` (notebook 04)
- `campaign/2_full/full_result.csv` (full campaign coding, one row per (game, trace, expert))
- `campaign/3_adjudication/adjudication_result.csv` (adjudication campaign export, standard FEP format, one row per (game, trace, arbiter))

### Outputs:
- `campaign/3_adjudication/adjudication_assignment.csv`: games to adjudicate, assigned to both arbiters
- `campaign/3_adjudication/adjudication_input.csv`: the two experts' codings of these games, extract of
  `full_result.csv` in the same format
- `campaign/3_adjudication/gold.csv`: gold standard, one row per feedback, input of notebook 07

## 1. Setup and imports

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import cohen_kappa_score

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Paths relative to the notebook: notebooks/ -> project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CAMPAIGN_DIR = PROJECT_ROOT / "campaign"
FORM_PATH = CAMPAIGN_DIR / "2_full" / "full_form.json"
FULL_ASSIGNMENT_PATH = CAMPAIGN_DIR / "2_full" / "full_campaign_assignment.csv"
FULL_RESULT_PATH = CAMPAIGN_DIR / "2_full" / "full_result.csv"
ADJ_DIR = CAMPAIGN_DIR / "3_adjudication"
ADJ_ASSIGNMENT_PATH = ADJ_DIR / "adjudication_assignment.csv"
ADJ_RESULT_PATH = ADJ_DIR / "adjudication_result.csv"
INPUT_PATH = ADJ_DIR / "adjudication_input.csv"
GOLD_PATH = ADJ_DIR / "gold.csv"

ARBITERS = ["Penny", "Shaquille"]   # pilot experts, not among the full campaign experts

print("Project root :", PROJECT_ROOT.name)
for path in [FORM_PATH, FULL_ASSIGNMENT_PATH, FULL_RESULT_PATH, ADJ_RESULT_PATH]:
    print(f"{str(path.relative_to(PROJECT_ROOT)):52s} exists: {path.exists()}")

## 2. Inputs

In [ ]:
import json

form = json.load(open(FORM_PATH, encoding="utf-8"))
QUESTIONS = form["questions"]
QUESTION_IDS = [q["id"] for q in QUESTIONS]
MULTI = {q["id"] for q in QUESTIONS if q["widget"]["type"] == "CHECKBOX"}
OTHER_COLS = [f"{q['id']}_other" for q in QUESTIONS if any(o.get("allowsFreeText") for o in q["widget"]["options"])]

assignment = pd.read_csv(FULL_ASSIGNMENT_PATH, dtype=str, keep_default_na=False)
full_result = pd.read_csv(FULL_RESULT_PATH, dtype=str, keep_default_na=False) if FULL_RESULT_PATH.exists() else None
adj_result = pd.read_csv(ADJ_RESULT_PATH, dtype=str, keep_default_na=False) if ADJ_RESULT_PATH.exists() else None

assert not set(ARBITERS) & set(assignment.expert_1) | set(ARBITERS) & set(assignment.expert_2), "an arbiter coded in the full campaign"
print(f"Full campaign: {len(assignment)} games, experts {sorted(set(assignment.expert_1) | set(assignment.expert_2))}")
print(f"Full campaign coding: {'%d rows' % len(full_result) if full_result is not None else 'not available yet'}")
print(f"Adjudication coding: {'%d rows' % len(adj_result) if adj_result is not None else 'not available yet'}")

## 3. Choices to adjudicate

For each feedback, the two codings are compared question by question; for the multiple-choice
questions, the comparison is on the set of ticked options (order does not matter); the free texts of
the "other" options are not compared. Each question in disagreement becomes one **choice** between the
two experts' answers. The platform applies the same rule to show the arbiters what to decide.

In [ ]:
def normalize(value: str, multi: bool) -> str:
    if not multi:
        return value.strip()
    return ",".join(sorted(v.strip() for v in value.split(",") if v.strip()))


def normalized(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in QUESTION_IDS:
        out[col] = out[col].map(lambda v, m=col in MULTI: normalize(v, m))
    return out


OTHER_OF = {q["id"]: f"{q['id']}_other" for q in QUESTIONS if f"{q['id']}_other" in OTHER_COLS}
choices = pd.DataFrame(columns=["game_id", "trace_id", "question_id", "a_expert", "a_value", "a_other",
                                "b_expert", "b_value", "b_other"])
if full_result is None:
    print("Choice detection skipped: full campaign coding not available yet.")
else:
    coded = normalized(full_result)
    first_expert = dict(zip(assignment.game_id, assignment.expert_1))
    n_coders = coded.groupby(["game_id", "trace_id"]).expert_id.nunique()
    assert (n_coders == 2).all(), n_coders[n_coders != 2]
    rows = []
    for (game_id, trace_id), group in coded.groupby(["game_id", "trace_id"]):
        group = group.sort_values("expert_id", key=lambda e: e != first_expert[game_id])
        a, b = group.iloc[0], group.iloc[1]
        for col in QUESTION_IDS:
            if a[col] != b[col]:
                other = OTHER_OF.get(col)
                rows.append({"game_id": game_id, "trace_id": trace_id, "question_id": col,
                             "a_expert": a.expert_id, "a_value": a[col], "a_other": a[other] if other else "",
                             "b_expert": b.expert_id, "b_value": b[col], "b_other": b[other] if other else ""})
    choices = pd.DataFrame(rows, columns=choices.columns)
    n_feedback = len(n_coders)
    n_disagree = choices.groupby(["game_id", "trace_id"]).ngroups
    print(f"Feedback coded: {n_feedback} | in agreement: {n_feedback - n_disagree} "
          f"| with at least one question to adjudicate: {n_disagree} ({n_disagree / n_feedback:.0%})")
    print("\nChoices to adjudicate, per question:")
    print(choices.question_id.value_counts().reindex(QUESTION_IDS, fill_value=0).to_string())

## 4. Files for the platform

Every game holding at least one choice is assigned to both arbiters:

- `adjudication_assignment.csv`: the games to adjudicate, in the format of the full campaign
  assignment (`game_id`, `group_id`, `n_received`, `expert_1`, `expert_2`), the two arbiters as experts;
- `adjudication_input.csv`: the two experts' codings of every feedback of these games, an extract of
  `full_result.csv` in the same format.

In [ ]:
if choices.empty:
    print("Files not written: no choice to adjudicate yet.")
    adj_assignment = pd.DataFrame()
else:
    adj_assignment = assignment[assignment.game_id.isin(choices.game_id)].copy()
    adj_assignment["expert_1"], adj_assignment["expert_2"] = ARBITERS
    adj_assignment = adj_assignment[["game_id", "group_id", "n_received", "expert_1", "expert_2"]]
    adj_input = full_result[full_result.game_id.isin(adj_assignment.game_id)]
    adj_input = adj_input.sort_values(["game_id", "trace_id", "expert_id"]).reset_index(drop=True)

    assert set(adj_assignment.game_id) == set(choices.game_id)
    per_game = choices.groupby("game_id").agg(feedback=("trace_id", "nunique"), choices=("question_id", "size"))
    print(f"Games to adjudicate: {len(adj_assignment)} of {len(assignment)} "
          f"(B={int((adj_assignment.group_id == 'B').sum())}, C={int((adj_assignment.group_id == 'C').sum())})")
    print(f"Per arbiter: {int(per_game.feedback.sum())} feedback with {int(per_game.choices.sum())} choices")
    adj_assignment.to_csv(ADJ_ASSIGNMENT_PATH, index=False, encoding="utf-8", lineterminator="\n")
    adj_input.to_csv(INPUT_PATH, index=False, encoding="utf-8", lineterminator="\n")
    print(f"Written: {ADJ_ASSIGNMENT_PATH.relative_to(PROJECT_ROOT)}, {INPUT_PATH.relative_to(PROJECT_ROOT)} ({len(adj_input)} rows)")
adj_assignment

## 5. Gold standard

The adjudication export has the standard FEP format (one row per (game, trace, arbiter), one column
per question). For each choice, the two arbiters' answers are compared with the same rule as in
section 3; the notebook also checks that each answer is one of the two experts' answers. Agreement
between the two arbiters is reported per question (raw agreement and Cohen's kappa).

In [ ]:
def free_text(values) -> str:
    texts = [t.strip() for t in values if t and t.strip()]
    return " | ".join(dict.fromkeys(texts))


gold = pd.DataFrame(columns=["game_id", "trace_id"] + QUESTION_IDS + OTHER_COLS + ["source"])
status = pd.Series(dtype=int)
if full_result is None:
    print("Gold standard skipped: full campaign coding not available yet.")
else:
    choice_keys = set(zip(choices.game_id, choices.trace_id, choices.question_id))
    options_of = {(r.game_id, r.trace_id, r.question_id): {r.a_value, r.b_value} for r in choices.itertuples()}
    decided, split_keys, outside = {}, set(), []
    if adj_result is not None:
        arb = normalized(adj_result[adj_result.expert_id.isin(ARBITERS)]).set_index(["game_id", "trace_id", "expert_id"])
        print("Arbiters' agreement per question:")
        for col in QUESTION_IDS:
            keys = [k for k in choice_keys if k[2] == col]
            pairs = []
            for game_id, trace_id, _ in keys:
                answers = [arb[col].get((game_id, trace_id, a)) for a in ARBITERS]
                if None in answers:
                    continue
                outside += [(game_id, trace_id, col, a) for a in answers if a not in options_of[(game_id, trace_id, col)]]
                pairs.append(answers)
                if answers[0] == answers[1]:
                    decided[(game_id, trace_id, col)] = answers[0]
                else:
                    split_keys.add((game_id, trace_id, col))
            if pairs:
                a, b = pd.Series([p[0] for p in pairs]), pd.Series([p[1] for p in pairs])
                kappa = 1.0 if (a == b).all() else cohen_kappa_score(a, b)
                print(f"  {col:22s} {len(pairs):4d} choices  raw agreement {(a == b).mean():.0%}  kappa {kappa:.2f}")
        if outside:
            print(f"WARNING: {len(outside)} arbiter answers are not one of the two experts' answers")

    rows = []
    blocked = {"split": set(), "open": set()}
    for (game_id, trace_id), group in coded.groupby(["game_id", "trace_id"]):
        record = {"game_id": game_id, "trace_id": trace_id}
        source = "experts"
        for col in QUESTION_IDS:
            key = (game_id, trace_id, col)
            other = OTHER_OF.get(col)
            if key not in choice_keys:
                record[col] = group[col].iloc[0]
                if other:
                    record[other] = free_text(group[other])
            elif key in decided:
                record[col] = decided[key]
                if other:
                    record[other] = free_text(group.loc[group[col] == decided[key], other])
                source = "adjudication"
            else:
                blocked["split" if key in split_keys else "open"].add((game_id, trace_id))
        if (game_id, trace_id) not in blocked["split"] | blocked["open"]:
            rows.append({**record, "source": source})

    gold = pd.DataFrame(rows, columns=gold.columns).sort_values(["game_id", "trace_id"]).reset_index(drop=True)
    assert not gold.duplicated(["game_id", "trace_id"]).any()
    status = pd.Series({
        "in agreement": int((gold.source == "experts").sum()),
        "adjudicated, kept": int((gold.source == "adjudication").sum()),
        "arbiters disagree, left out": len(blocked["split"]),
        "not adjudicated yet": len(blocked["open"] - blocked["split"]),
    })
    print("\nFeedback:")
    print(status.to_string())
    gold.to_csv(GOLD_PATH, index=False, encoding="utf-8", lineterminator="\n")
    print(f"\nWritten: {GOLD_PATH.relative_to(PROJECT_ROOT)} ({len(gold)} feedback)")

## 6. Summary

In [ ]:
summary = pd.DataFrame([
    {"step": "Feedback coded by two experts", "value": 0 if full_result is None else int(n_feedback)},
    {"step": "Choices to adjudicate (question x feedback)", "value": len(choices)},
    {"step": "Games sent to adjudication", "value": len(adj_assignment)},
    {"step": "Arbiters", "value": ", ".join(ARBITERS)},
    {"step": "Gold standard (feedback)", "value": len(gold)},
    {"step": "  - from expert agreement", "value": 0 if gold.empty else int((gold.source == "experts").sum())},
    {"step": "  - from adjudication", "value": 0 if gold.empty else int((gold.source == "adjudication").sum())},
    {"step": "Left out (arbiters disagree)", "value": int(status.get("arbiters disagree, left out", 0))},
])
summary